# 🛰️ GeoCrop en WebAssembly — series de tiempo satelitales sin servidor

Este cuaderno ejecuta la **generación de series de tiempo** de [geocrop_analysis_mx](https://github.com/abxda/geocrop_analysis_mx)
—búsqueda STAC, lectura de COGs por rangos HTTP, compuestos geomediana y segmentación Shepherd—
**directamente en tu navegador** vía [Pyodide](https://pyodide.org) (WebAssembly), o idéntico en CPython local.

Sin Google Earth Engine, sin conda, sin servidor:

| Insumo | Fuente (en navegador) | Acceso |
|---|---|---|
| Óptico Sentinel-2 L2A | Earth Search (Element 84 / AWS) | anónimo, CORS ✓ |
| Radar Sentinel-1 RTC (gamma-0) | Microsoft Planetary Computer | anónimo (token SAS automático), CORS ✓ |
| Segmentación Shepherd | [shepherd-wasm](https://github.com/abxda/shepherd-wasm) | pip / micropip |

> En CPython (fuera del navegador) el pipeline completo usa además **HLS** (NASA LPCLOUD o
> Planetary Computer) vía `odc-stac`; en el navegador se usa Sentinel-2 de Earth Search porque
> es la fuente óptica con CORS habilitado.

In [ ]:
# ¿Dónde está corriendo este Python?
import sys, platform
print(f"Python {sys.version.split()[0]} · plataforma {sys.platform!r} · arquitectura {platform.machine()!r}")
if sys.platform == "emscripten":
    print("✅ Corriendo en WebAssembly, DENTRO de tu navegador. No hay servidor. 🚀")
else:
    print("ℹ️ Corriendo en modo local (CPython). El código es exactamente el mismo.")

In [ ]:
# Paso 0 — Instalar dependencias (pequeñas y puras: micropip las resuelve en el navegador)
%pip install -q pystac-client planetary-computer odc-geo tifffile shepherd-wasm pyodide-http requests

In [ ]:
# Paso 1 — Traer los módulos del pipeline (los mismos archivos del repositorio)
import os, sys

RAW = "https://raw.githubusercontent.com/abxda/geocrop_analysis_mx/main/src/data_download/"
MODULES = ["__init__.py", "stac_utils.py", "stac_multispectral.py", "stac_radar.py", "cog_fetch.py"]

os.makedirs("data_download", exist_ok=True)

async def fetch_text(url):
    if sys.platform == "emscripten":
        from pyodide.http import pyfetch
        return await (await pyfetch(url)).text()
    import requests
    return requests.get(url, timeout=60).text

for name in MODULES:
    if not os.path.exists(f"data_download/{name}"):
        open(f"data_download/{name}", "w").write(await fetch_text(RAW + name))
sys.path.insert(0, ".")

from data_download import stac_utils, stac_multispectral, stac_radar, cog_fetch
stac_utils.configure_gdal_env()   # en el navegador también enruta requests -> fetch
print("Módulos del pipeline listos.")

In [ ]:
# Paso 2 — Definir el área de interés (Valle del Yaqui, Sonora) y la malla de salida
AOI_BBOX = (-109.78, 27.26, -109.68, 27.34)   # ~10 x 9 km — cámbialo por tu zona
MES      = "2023-03"                            # mes a componer

from odc.geo.geobox import GeoBox
geobox = GeoBox.from_bbox(AOI_BBOX, crs="EPSG:4326",
                          resolution=30 / stac_utils.METERS_PER_DEGREE)
aoi = {"type": "Polygon", "coordinates": [[
    [AOI_BBOX[0], AOI_BBOX[1]], [AOI_BBOX[2], AOI_BBOX[1]],
    [AOI_BBOX[2], AOI_BBOX[3]], [AOI_BBOX[0], AOI_BBOX[3]],
    [AOI_BBOX[0], AOI_BBOX[1]]]]}
import calendar
y, m = map(int, MES.split("-"))
inicio, fin = f"{MES}-01", f"{MES}-{calendar.monthrange(y, m)[1]:02d}"
print(f"Malla de salida: {geobox.width} x {geobox.height} px a 30 m · {inicio} → {fin}")

In [ ]:
# Paso 3 — Óptico: buscar escenas, leer solo los tiles del AOI y componer la geomediana
# ("auto" elige Earth Search S2 L2A en el navegador; HLS NASA/MPC en CPython)
proveedor, catalogo = stac_multispectral.resolve_provider("auto")
items = stac_multispectral.search_hls_items(catalogo, inicio, fin, aoi, proveedor)
print("Escenas encontradas:", {k: len(v) for k, v in items.items()})

stack = stac_multispectral.load_hls_stack(items, geobox, proveedor)
print(f"Observaciones (días solares): {stack.sizes['time']}")

compuesto = stac_multispectral.build_composite(stack)   # (13, y, x): 6 bandas + 7 índices, geomediana Weiszfeld
print("Compuesto listo:", compuesto.shape)

In [ ]:
# Paso 4 — Visualizar RGB y NDVI de la geomediana
import numpy as np
import matplotlib.pyplot as plt

rgb = np.dstack([compuesto[2], compuesto[1], compuesto[0]]) / 3000.0
ndvi = compuesto[6] / 10000.0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.imshow(np.clip(rgb, 0, 1)); ax1.set_title(f"RGB geomediana {MES}"); ax1.axis("off")
im = ax2.imshow(ndvi, cmap="RdYlGn", vmin=0, vmax=0.9)
ax2.set_title(f"NDVI {MES}"); ax2.axis("off")
plt.colorbar(im, ax=ax2, shrink=0.8); plt.tight_layout(); plt.show()

In [ ]:
# Paso 5 — Radar Sentinel-1 RTC: cada escena pesa ~1.9 GB, pero el lector por
# rangos HTTP trae únicamente los tiles que tocan el AOI (unos cuantos MB)
catalogo_radar = stac_utils.get_catalog()   # Planetary Computer, anónimo
items_s1 = stac_radar.search_s1_items(catalogo_radar, inicio, fin, aoi)
print(f"Escenas S1 RTC: {len(items_s1)}")

stack_s1 = stac_radar.load_s1_stack(items_s1, geobox)
radar = stac_radar.build_composite(stack_s1)   # (3, y, x): VV dB, VH dB, RVI — mediana mensual
print("Compuesto radar listo:", radar.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.imshow(radar[0], cmap="gray", vmin=-20, vmax=0); ax1.set_title("VV (dB, gamma-0)"); ax1.axis("off")
ax2.imshow(radar[1], cmap="gray", vmin=-28, vmax=-8); ax2.set_title("VH (dB, gamma-0)"); ax2.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Paso 6 — Segmentación Shepherd (shepherd-wasm: NumPy/SciPy puro, sin numba)
import shepherd_wasm

img = np.nan_to_num(compuesto, nan=0).astype(np.int16)   # (bandas, filas, cols)
resultado = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=40, minSegmentSize=50, imgNullVal=0)
seg = resultado.segimg
print(f"Segmentos: {len(np.unique(seg))}")

from scipy import ndimage
bordes = (ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2))
vis = np.clip(rgb, 0, 1).copy(); vis[bordes] = [1, 1, 0]
plt.figure(figsize=(9, 7)); plt.imshow(vis)
plt.title("Parcelas segmentadas sobre la geomediana"); plt.axis("off"); plt.show()

## ¿Qué acaba de pasar?

1. **Búsqueda STAC** anónima (Earth Search / Planetary Computer) — solo metadatos JSON.
2. **Lectura quirúrgica de COGs**: en CPython, `odc-stac` hace lecturas por ventana vía GDAL;
   en el navegador, `cog_fetch.py` replica eso con peticiones HTTP `Range` + `tifffile`
   (incluye un decodificador NumPy puro del predictor de punto flotante de TIFF, que
   normalmente requiere la extensión C `imagecodecs`).
3. **Geomediana multivariante** (Weiszfeld, NumPy puro) — el mismo estadístico robusto de GEE/DEA.
4. **Segmentación Shepherd** con `shepherd-wasm`, bit-exacta contra `pyshepseg`.

### Límites conocidos en el navegador
- **HLS (NASA/MPC) no funciona en WASM**: el blob de Azure de HLS no envía cabeceras CORS y
  LPCLOUD requiere token con redirecciones que el sandbox no permite. Por eso `auto` usa
  Sentinel-2 L2A de Earth Search (CORS ✓). El pipeline CPython sí usa HLS.
- La memoria del navegador (~2 GB en Pyodide) limita el AOI y el número de meses por sesión.
- Las fases de ML del pipeline (TPOT) no corren en WASM; para clasificación en navegador usa
  scikit-learn directo (ver el [taller WASM](https://github.com/abxda/portable-satelital)).